# 00 · Data acquisition

This notebook downloads the experiment data used in the project and checks it against published figures before anything else runs.

| Dataset | Used for | Status |
|---|---|---|
| Hillstrom MineThatData email test (64,000 customers, three randomised arms) | Notebooks 01 and 02: diagnostics, effect estimation, uplift models with interpretable features | Downloaded here |
| Criteo uplift dataset (about 14M rows, anonymised features) | Later notebooks: effect heterogeneity at scale, Qini evaluation, compliance-adjusted effects | Added when we reach that notebook |

**Source and terms.** The Hillstrom data was published openly by Kevin Hillstrom (MineThatData) for his 2008 E-Mail Analytics and Data Mining Challenge. No licence file accompanies it, so credit the source wherever results are shown. Raw data is excluded from version control (see `.gitignore`); anyone cloning the repository runs this notebook to fetch it.

In [1]:
import hashlib, json, os, urllib.request
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

OFFICIAL_URL = (
    "http://www.minethatdata.com/"
    "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
)
# Set HILLSTROM_SOURCE_URL to point at a mirror or a local file:// path if the official host is unreachable.
SOURCE_URL = os.environ.get("HILLSTROM_SOURCE_URL", OFFICIAL_URL)

RAW_DIR = Path("../data/raw")
DEST = RAW_DIR / "hillstrom.csv"
MANIFEST = RAW_DIR / "MANIFEST.json"

# Published figures to verify against (Hillstrom challenge description; arm sizes from the challenge write-ups)
EXPECTED_ROWS = 64_000
EXPECTED_COLUMNS = [
    "recency", "history_segment", "history", "mens", "womens", "zip_code",
    "newbie", "channel", "segment", "visit", "conversion", "spend",
]
EXPECTED_ARMS = {"Mens E-Mail": 21_307, "Womens E-Mail": 21_387, "No E-Mail": 21_306}

In [2]:
def download(url: str, dest: Path, timeout: int = 60) -> None:
    """Download to a temporary file, then move into place so a failed download never leaves a partial file."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(".part")
    request = urllib.request.Request(url, headers={"User-Agent": "uplift-experimentation-framework/0.1"})
    with urllib.request.urlopen(request, timeout=timeout) as response, open(tmp, "wb") as fh:
        fh.write(response.read())
    tmp.replace(dest)


def verify(path: Path) -> dict:
    """Check the file against the published figures. Returns a dict of check name -> bool."""
    df = pd.read_csv(path)
    arm_counts = df["segment"].value_counts().to_dict() if "segment" in df else {}
    return {
        "columns match": list(df.columns) == EXPECTED_COLUMNS,
        f"row count is {EXPECTED_ROWS:,}": len(df) == EXPECTED_ROWS,
        "arm sizes match published figures": arm_counts == EXPECTED_ARMS,
        "no missing values": int(df.isna().sum().sum()) == 0,
    }


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

In [3]:
if DEST.exists():
    print(f"{DEST} already exists - skipping download.")
else:
    try:
        download(SOURCE_URL, DEST)
        print(f"Downloaded {DEST.stat().st_size / 1e6:.1f} MB from {SOURCE_URL}")
    except Exception as exc:  # network errors, DNS, blocked hosts
        raise SystemExit(
            f"Download failed: {exc}\n"
            f"Fetch the file manually from:\n  {OFFICIAL_URL}\n"
            f"and save it as {DEST.resolve()}, or set HILLSTROM_SOURCE_URL to a reachable copy."
        )

Downloaded 4.0 MB from http://www.minethatdata.com/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv


In [4]:
checks = verify(DEST)
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values()), f"Verification failed. Delete {DEST} and re-run this notebook to download a fresh copy."

MANIFEST.write_text(json.dumps({
    "file": DEST.name,
    "source_url": SOURCE_URL,
    "retrieved_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "sha256": sha256(DEST),
    "rows": EXPECTED_ROWS,
    "checks_passed": list(checks),
}, indent=2))
print(f"\nManifest written to {MANIFEST}")

PASS  columns match
PASS  row count is 64,000
PASS  arm sizes match published figures
PASS  no missing values

Manifest written to ..\data\raw\MANIFEST.json


## Notes

- The source file spells one `zip_code` category as `Surburban`. Notebook 01 corrects the label to `Suburban` after loading; the raw file is left untouched.
- The file has no customer identifier. Notebook 01 examines what that means for duplicate rows.
- The manifest records the SHA-256 of the file used, so results can be tied to an exact copy of the data.